# HW2 Digit Enhance (Clean)

Digit MNIST experiments with HOG features, augmentation, and model improvements.

In [ ]:
import os
import copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
from skimage.feature import hog

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Data loading
PROJECT_DATA_DIR = './data'
os.makedirs(PROJECT_DATA_DIR, exist_ok=True)
os.environ['KERAS_HOME'] = PROJECT_DATA_DIR

from tensorflow.keras.datasets import mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()
print('Digit MNIST loaded:', X_train.shape, y_train.shape)

In [ ]:
def describe_raw(name, X, y):
    print(f'{name}')
    print(f'  X shape: {X.shape}, dtype: {X.dtype}')
    print(f'  y shape: {y.shape}, dtype: {y.dtype}')
    print(f'  X min/max: {X.min()} / {X.max()}')
    print(f'  Unique labels: {np.unique(y)}')

describe_raw('Digit MNIST (raw)', X_train, y_train)

In [ ]:
def show_samples(title, images, labels=None, n=8, cmap='gray'):
    n = min(n, len(images))
    plt.figure(figsize=(n * 2, 2.5))
    for i in range(n):
        plt.subplot(1, n, i + 1)
        plt.imshow(images[i], cmap=cmap)
        if labels is not None:
            plt.title(str(labels[i]))
        plt.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

## Feature Engineering 1: HOG

In [ ]:
def extract_hog_features(images):
    images = images.reshape(-1, 28, 28)
    win_size = images[0].shape
    cell_size = (4, 4)
    block_size = (8, 8)
    block_stride = (4, 4)
    num_bins = 9
    hog_desc = cv2.HOGDescriptor(win_size, block_size, block_stride, cell_size, num_bins)
    num_images = images.shape[0]
    feat_size = len(hog_desc.compute(images[0].astype(np.uint8)))
    feats = np.zeros((num_images, feat_size), dtype=np.float32)
    for i, img in enumerate(images):
        feats[i] = hog_desc.compute(img.astype(np.uint8)).flatten()
    return feats

def show_hog_visuals(images, n=6):
    n = min(n, len(images))
    plt.figure(figsize=(n * 2.5, 2.5))
    for i in range(n):
        _, hog_image = hog(images[i], orientations=9, pixels_per_cell=(4, 4),
                           cells_per_block=(2, 2), block_norm='L2-Hys', visualize=True)
        plt.subplot(1, n, i + 1)
        plt.imshow(hog_image, cmap='gray')
        plt.axis('off')
    plt.suptitle('HOG visualization')
    plt.tight_layout()
    plt.show()

show_hog_visuals(X_train[:6])

In [ ]:
X_train_hog = extract_hog_features(X_train)
X_test_hog = extract_hog_features(X_test)

### Nhan xet: HOG
- Ghi metrics: Accuracy __, F1 __, Precision __, Recall __.
- HOG mo ta huong gradient, giup tach biet cac chu so co net tuong dong.

## Model + Training

In [ ]:
class BaselineANN(nn.Module):
    def __init__(self, input_size, hidden_size=512, output_size=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size // 2)
        self.fc3 = nn.Linear(hidden_size // 2, 128)
        self.fc4 = nn.Linear(128, output_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.fc4(x)

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0
    for X, y in dataloader:
        X = X.to(device)
        y = y.to(device)
        preds = model(X)
        loss = criterion(preds, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        _, predicted = torch.max(preds, 1)
        acc = (predicted == y).sum().item() / len(y)
        total_loss += loss.item()
        total_acc += acc
        num_batches += 1
    return total_loss / num_batches, total_acc / num_batches

def train_model(model, dataloader, epochs=20, lr=0.001, save_best=None):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best_loss = float('inf')
    for epoch in range(epochs):
        loss, acc = train_epoch(model, dataloader, optimizer, criterion)
        print(f'Epoch {epoch+1}/{epochs} - Loss: {loss:.4f}, Acc: {acc:.4f}')
        if save_best and loss < best_loss:
            best_loss = loss
            torch.save(copy.deepcopy(model.state_dict()), save_best)
    return model

def evaluate_model(model, X_test, y_test, num_classes=10):
    model.eval()
    with torch.no_grad():
        inputs = torch.tensor(X_test, dtype=torch.float32).to(device)
        labels = torch.tensor(y_test).to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        pred_cpu = predicted.cpu().numpy()
        labels_cpu = labels.cpu().numpy()
        accuracy = (pred_cpu == labels_cpu).mean()
        f1 = f1_score(labels_cpu, pred_cpu, average='weighted')
        precision = precision_score(labels_cpu, pred_cpu, average='weighted')
        recall = recall_score(labels_cpu, pred_cpu, average='weighted')
        cm = confusion_matrix(labels_cpu, pred_cpu)
        print(f'Accuracy: {accuracy:.4f}')
        print(f'F1 Score: {f1:.4f}')
        print(f'Precision: {precision:.4f}')
        print(f'Recall: {recall:.4f}')
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title('Confusion Matrix')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.show()
        return accuracy, f1, precision, recall, cm

In [ ]:
hog_loader = DataLoader(list(zip(torch.tensor(X_train_hog, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))),
                        batch_size=4096, shuffle=True, num_workers=2)
hog_model = BaselineANN(input_size=X_train_hog.shape[1]).to(device)
# train_model(hog_model, hog_loader, epochs=30, lr=0.001, save_best='best_digit_hog.pth')
# evaluate_model(hog_model, X_test_hog, y_test, num_classes=10)

### Nhan xet: Baseline Model (HOG input)
- Ghi metrics: Accuracy __, F1 __, Precision __, Recall __.
- Neu gap nham lan cap chu so (vd 3-9, 4-9), ghi ro vi du tu confusion matrix.

## Feature Engineering 2: Data Augmentation (HOG from augmented images)

In [ ]:
class HOGAugDataset(Dataset):
    def __init__(self, data, labels, transform=None):
        if len(data.shape) == 2:
            self.data = data.reshape(-1, 28, 28).astype(np.uint8)
        else:
            self.data = data.astype(np.uint8)
        self.labels = labels
        self.transform = transform
        win_size = (28, 28)
        cell_size = (4, 4)
        block_size = (8, 8)
        block_stride = (4, 4)
        num_bins = 9
        self.hog = cv2.HOGDescriptor(win_size, block_size, block_stride, cell_size, num_bins)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = self.data[idx]
        if self.transform:
            image = self.transform(transforms.ToPILImage()(image))
            image = np.array(image, dtype=np.uint8)
        hog_feat = self.hog.compute(image).flatten()
        return torch.tensor(hog_feat, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)

aug_transform = transforms.Compose([
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.RandomAffine(degrees=0, scale=(0.9, 1.1)),
])

aug_dataset = HOGAugDataset(X_train, y_train, transform=aug_transform)
aug_loader = DataLoader(aug_dataset, batch_size=4096, shuffle=True, num_workers=2)

# Plot augmented examples
aug_images = []
for i in range(6):
    img = np.array(aug_transform(transforms.ToPILImage()(X_train[i])), dtype=np.uint8)
    aug_images.append(img)
show_samples('Augmented samples', np.array(aug_images))

In [ ]:
aug_model = BaselineANN(input_size=X_train_hog.shape[1]).to(device)
# train_model(aug_model, aug_loader, epochs=20, lr=0.001, save_best='best_digit_aug.pth')
# evaluate_model(aug_model, X_test_hog, y_test, num_classes=10)

### Nhan xet: HOG + Data Augmentation
- Ghi metrics: Accuracy __, F1 __, Precision __, Recall __.
- Augmentation lam phong phu goc nhin, thuong tang do on dinh.

## Model Improvements (BatchNorm + Dropout)

In [ ]:
class ANNWithBNDropout(nn.Module):
    def __init__(self, input_size, hidden_size=512, output_size=10, dropout_rate=0.2):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.bn1 = nn.BatchNorm1d(hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size // 2)
        self.bn2 = nn.BatchNorm1d(hidden_size // 2)
        self.fc3 = nn.Linear(hidden_size // 2, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.fc4 = nn.Linear(128, output_size)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.dropout(F.relu(self.bn1(self.fc1(x))))
        x = self.dropout(F.relu(self.bn2(self.fc2(x))))
        x = F.relu(self.bn3(self.fc3(x)))
        return self.fc4(x)

bn_do_model = ANNWithBNDropout(input_size=X_train_hog.shape[1]).to(device)
# train_model(bn_do_model, aug_loader, epochs=40, lr=0.01, save_best='best_digit_bn_do.pth')
# evaluate_model(bn_do_model, X_test_hog, y_test, num_classes=10)

### Nhan xet: BatchNorm + Dropout
- Ghi metrics: Accuracy __, F1 __, Precision __, Recall __.
- BN giup hoi tu nhanh, Dropout giam overfit.

### Tong hop nhanh
| Phuong phap | Accuracy | F1 | Precision | Recall |
|---|---:|---:|---:|---:|
| HOG | __ | __ | __ | __ |
| HOG + Aug | __ | __ | __ | __ |
| HOG + BN/Dropout | __ | __ | __ | __ |

## Notes

- Record metrics for HOG, augmented HOG, and BN+Dropout variants.
- Include confusion matrices for the final model in the report.